# 22-32 · Параметризация против инъекции

Практика к разделу [«Основы безопасности веб-приложений»](../../site/chapters/glava-22/22-32-bezopasnost.html).

## Цель

На настоящем sqlite3 сравнить опасную сборку запроса через f-строку с безопасным параметризованным запросом — и увидеть, к чему приводит SQL-инъекция, если её не остановить.

## Рабочий пример — уязвимый вариант

Используем одноинструкционную инъекцию, которую принимает обычный `sqlite3.execute()`. Составную атаку с `DROP TABLE` этот метод отклонил бы, но это ограничение Python-драйвера не делает SQL из f-строки безопасным.

In [ ]:
import sqlite3

baza = sqlite3.connect(":memory:")
baza.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
zadachi = [("Купить хлеб",), ("Написать тесты",), ("Позвонить врачу",)]
baza.executemany("INSERT INTO tasks (title) VALUES (?)", zadachi)
baza.commit()

vvod_polzovatelya = "' OR '1'='1"

# ТАК ДЕЛАТЬ НЕЛЬЗЯ — показано специально, чтобы увидеть последствия:
opasny_zapros = f"SELECT * FROM tasks WHERE title = '{vvod_polzovatelya}'"
rezultat_opasny = baza.execute(opasny_zapros).fetchall()

print("Введённый фильтр:", repr(vvod_polzovatelya))
print("Строки, возвращённые уязвимым запросом:", rezultat_opasny)

## Проверка результата — фильтр обойдён

In [ ]:
vse_stroki = baza.execute("SELECT * FROM tasks ORDER BY id").fetchall()
assert rezultat_opasny == vse_stroki
assert len(rezultat_opasny) == len(zadachi) == 3
print(f"Фильтр обойдён: запрос вернул все {len(rezultat_opasny)} строки вместо точного совпадения.")

## Эксперимент — параметризованный запрос безопасен

In [ ]:
baza2 = sqlite3.connect(":memory:")
baza2.execute("CREATE TABLE tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL)")
baza2.executemany("INSERT INTO tasks (title) VALUES (?)", zadachi)
baza2.commit()

vvod_kak_dannye = "' OR '1'='1"
rezultat = baza2.execute("SELECT * FROM tasks WHERE title = ?", (vvod_kak_dannye,)).fetchall()

vse_stroki2 = baza2.execute("SELECT * FROM tasks ORDER BY id").fetchall()
print("Строки по точному совпадению:", rezultat)
print("Все строки по-прежнему в таблице:", vse_stroki2)

assert rezultat == []
assert len(vse_stroki2) == len(zadachi) == 3
print("Верно: параметризованный запрос обработал вредоносный текст как обычное значение и не позволил обойти фильтр.")